# Task 2: Advanced EDA & Text Mining
Pipeline: load -> assess -> clean -> stats/plots -> entity tagging -> issue classification -> TF-IDF + KMeans clustering -> root causes -> export.

The notebook auto-detects the free-text and date columns. **Set `TEXT_COL`, `DATE_COL` etc. manually in the config cell if detection picks wrong.**

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn nltk openpyxl wordcloud

In [ ]:
import re, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from collections import Counter
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

df = pd.read_csv("Hanuai task2 assignment.csv", encoding_errors="replace")
print(df.shape); df.head()

## 1. Data assessment

In [ ]:
df.info()
print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False))
print("\nMissing %:\n", (df.isna().mean()*100).round(1).sort_values(ascending=False))
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nUnique values per column:\n", df.nunique().sort_values())
df.describe(include="all").T

In [ ]:
# ---- CONFIG: auto-detect, override if needed ----
obj_cols = df.select_dtypes(include="object").columns
avg_len = {c: df[c].dropna().astype(str).str.len().mean() for c in obj_cols}
TEXT_COL = max(avg_len, key=avg_len.get) if avg_len else None   # longest avg string = narrative field

DATE_COL = None
for c in df.columns:
    if re.search(r"date|time|created|reported|opened", c, re.I):
        parsed = pd.to_datetime(df[c], errors="coerce")
        if parsed.notna().mean() > 0.6: DATE_COL = c; break

# TEXT_COL = "Problem Description"      # <- override here if needed
# DATE_COL = "Date"
print("Text column:", TEXT_COL, "| Date column:", DATE_COL)
print(df[TEXT_COL].dropna().head(5).tolist())

## 2. Cleaning

In [ ]:
d = df.copy()
d.columns = [c.strip() for c in d.columns]
d = d.drop_duplicates().reset_index(drop=True)

# strip whitespace + standardise categorical casing
for c in d.select_dtypes(include="object").columns:
    d[c] = d[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)

cat_cols = [c for c in d.select_dtypes(include=["object","string"]).columns
            if c != TEXT_COL and d[c].nunique() <= max(30, 0.05*len(d))]
for c in cat_cols:
    d[c] = d[c].str.title()

if DATE_COL:
    d[DATE_COL] = pd.to_datetime(d[DATE_COL], errors="coerce")

# imputation: median for numeric, mode/'Unknown' for categorical, '' for text
num_cols = d.select_dtypes(include=np.number).columns.tolist()
for c in num_cols: d[c] = d[c].fillna(d[c].median())
for c in cat_cols: d[c] = d[c].fillna(d[c].mode().iloc[0] if d[c].notna().any() else "Unknown")
d[TEXT_COL] = d[TEXT_COL].fillna("")
d = d[d[TEXT_COL].str.len() > 3].reset_index(drop=True)   # rows with no usable narrative
print("clean shape:", d.shape)

## 3. Summary statistics & visualisations

In [ ]:
if num_cols:
    display(d[num_cols].agg(["mean","median","std","min","max"]).T.round(2))

for c in cat_cols[:6]:
    top = d[c].value_counts().head(15)
    plt.figure(figsize=(9, 4)); sns.barplot(x=top.values, y=top.index, color="steelblue")
    plt.title(f"Frequency: {c}"); plt.xlabel("count"); plt.tight_layout(); plt.show()

for c in num_cols[:4]:
    plt.figure(figsize=(7, 3.5)); sns.histplot(d[c], kde=True); plt.title(f"Distribution: {c}"); plt.show()

if DATE_COL:
    ts = d.set_index(DATE_COL).resample("M").size()
    ts.plot(figsize=(10, 3.5), marker="o", title="Complaints per month"); plt.show()

## 4. Text mining & entity tagging (rule/dictionary-based NER)
Edit the dictionaries after reading a sample of your narratives - this is where domain tuning pays off.

In [ ]:
COMPONENTS = {
    "Battery": r"batter|charg(?:e|ing|er)|cell",
    "Motor": r"motor|engine|compressor",
    "Screen/Display": r"screen|display|lcd|touch ?screen|panel",
    "Sensor": r"sensor|probe|thermistor",
    "Control Board/PCB": r"\bpcb\b|board|controller|circuit",
    "Wiring/Connector": r"wire|wiring|cable|connector|harness|plug",
    "Fan/Cooling": r"\bfan\b|cooling|overheat|heat sink",
    "Pump/Valve/Hose": r"pump|valve|hose|pipe|seal|gasket",
    "Door/Hinge/Housing": r"door|hinge|latch|housing|cover|casing|handle",
    "Software/Firmware": r"software|firmware|app\b|update|bug|crash|freez|reboot",
}
SYMPTOMS = {
    "Not Working/Dead": r"not (?:work|turn|power|start)|won'?t (?:turn|start|power)|dead|no power|stopped working",
    "Leak": r"leak|drip|water on",
    "Noise": r"noise|noisy|loud|rattl|grind|buzz|squeal|clicking",
    "Overheating": r"overheat|too hot|burning|smoke|smell",
    "Intermittent": r"intermittent|randomly|sometimes|on and off|flicker",
    "Physical Damage": r"crack|broke|broken|dent|bent|loose|detach|scratch",
    "Display Issue": r"blank|black screen|flicker|dim|no display|dead pixel",
    "Error/Fault": r"error|fault|alarm|warning|code",
    "Slow/Performance": r"slow|lag|poor performance|weak|low (?:output|pressure|power)",
}
ERROR_CODE_RE = re.compile(r"\b(?:E|ERR|ERROR|F|FAULT|code)[\s\-:#]?\d{1,4}\b|\b[A-Z]{1,2}\d{2,4}\b", re.I)

def extract(text):
    t = str(text).lower()
    comps = [k for k, p in COMPONENTS.items() if re.search(p, t)]
    syms  = [k for k, p in SYMPTOMS.items() if re.search(p, t)]
    codes = sorted({m.group(0).upper().replace(" ", "") for m in ERROR_CODE_RE.finditer(str(text))})
    return pd.Series({"components": comps, "symptoms": syms, "error_codes": codes})

d = pd.concat([d, d[TEXT_COL].apply(extract)], axis=1)
print("rows with >=1 component:", (d.components.str.len() > 0).mean().round(3))
d[[TEXT_COL, "components", "symptoms", "error_codes"]].head(8)

## 5. Issue classification (high-level categories)

In [ ]:
CATEGORY_MAP = {
    "Component Failure": {"Battery","Motor","Sensor","Fan/Cooling","Pump/Valve/Hose","Screen/Display"},
    "Electrical Issue": {"Control Board/PCB","Wiring/Connector"},
    "Software Glitch": {"Software/Firmware"},
    "Assembly / Build Quality": {"Door/Hinge/Housing"},
}
SYMPTOM_CATEGORY = {"Physical Damage": "Assembly / Build Quality", "Overheating": "Electrical Issue",
                    "Error/Fault": "Software Glitch", "Leak": "Component Failure", "Noise": "Component Failure"}

def classify(row):
    scores = Counter()
    for c in row.components:
        for cat, members in CATEGORY_MAP.items():
            if c in members: scores[cat] += 2
    for s in row.symptoms:
        if s in SYMPTOM_CATEGORY: scores[SYMPTOM_CATEGORY[s]] += 1
    return scores.most_common(1)[0][0] if scores else "Other / Unclassified"

d["issue_category"] = d.apply(classify, axis=1)
vc = d.issue_category.value_counts()
vc.plot.barh(figsize=(8, 3.5), title="Issue categories"); plt.show(); print(vc)

# top components / symptoms / codes
for col in ["components", "symptoms", "error_codes"]:
    print(f"\nTop {col}:", Counter(x for l in d[col] for x in l).most_common(10))

## 6. Temporal / category trends

In [ ]:
if DATE_COL:
    d["month"] = d[DATE_COL].dt.to_period("M").astype(str)
    piv = d.pivot_table(index="month", columns="issue_category", values=TEXT_COL, aggfunc="count", fill_value=0)
    piv.plot(figsize=(11, 4), marker="o", title="Issue categories over time"); plt.show()
for c in cat_cols[:3]:
    ct = pd.crosstab(d[c], d.issue_category)
    if ct.shape[0] <= 25:
        plt.figure(figsize=(9, 5)); sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrRd"); plt.title(f"{c} x Issue category"); plt.show()

## 7. Clustering: TF-IDF + KMeans (k chosen by silhouette)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD

def prep(t):
    t = re.sub(r"[^a-z0-9\s]", " ", str(t).lower())
    return re.sub(r"\s+", " ", t).strip()

d["clean_text"] = d[TEXT_COL].map(prep)
stop = list(ENGLISH_STOP_WORDS | {"customer","called","stated","says","said","unit","product","issue","problem"})
vec = TfidfVectorizer(stop_words=stop, ngram_range=(1, 2), min_df=3, max_df=0.7, max_features=5000)
X = vec.fit_transform(d.clean_text)
print("TF-IDF matrix:", X.shape)

Xr = TruncatedSVD(n_components=min(100, X.shape[1]-1), random_state=42).fit_transform(X)
scores = {}
for k in range(3, 11):
    lab = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xr)
    scores[k] = silhouette_score(Xr, lab)
best_k = max(scores, key=scores.get)
plt.plot(list(scores), list(scores.values()), marker="o"); plt.title(f"Silhouette by k (best={best_k})"); plt.show()

km = KMeans(n_clusters=best_k, n_init=20, random_state=42).fit(X)
d["cluster"] = km.labels_

## 8. Root-cause analysis per cluster

In [ ]:
terms = np.array(vec.get_feature_names_out())
order = km.cluster_centers_.argsort()[:, ::-1]
summary = []
for k in range(best_k):
    sub = d[d.cluster == k]
    top_terms = ", ".join(terms[order[k, :10]])
    top_comp = Counter(x for l in sub.components for x in l).most_common(3)
    top_sym  = Counter(x for l in sub.symptoms for x in l).most_common(3)
    summary.append({"cluster": k, "size": len(sub), "share_%": round(100*len(sub)/len(d), 1),
                    "top_terms": top_terms, "top_components": top_comp, "top_symptoms": top_sym,
                    "dominant_issue_category": sub.issue_category.mode().iloc[0]})
cluster_summary = pd.DataFrame(summary).sort_values("size", ascending=False)
pd.set_option("display.max_colwidth", 120); display(cluster_summary)

for k in cluster_summary.cluster.head(3):
    print(f"\n--- Cluster {k} sample complaints ---")
    for s in d[d.cluster == k][TEXT_COL].head(3): print(" *", s[:200])

In [ ]:
# 2-D view of clusters
xy = TruncatedSVD(n_components=2, random_state=42).fit_transform(X)
plt.figure(figsize=(8, 6)); sns.scatterplot(x=xy[:, 0], y=xy[:, 1], hue=d.cluster, palette="tab10", s=18)
plt.title("Complaint clusters (SVD projection)"); plt.show()

## 9. Export
Fill in the stakeholder-report points using `cluster_summary` and the plots above: top failure modes, their share, probable root cause per cluster, estimated impact (warranty cost, returns), and next steps (supplier QA, firmware fix, assembly check). Future improvements: BERTopic / sentence-transformer embeddings, LLM-based entity extraction, labelled validation sample.

In [ ]:
out = d.copy()
for c in ["components","symptoms","error_codes"]:
    out[c] = out[c].map(lambda x: "; ".join(x))
out.drop(columns=["clean_text"]).to_csv("task2_enriched_dataset.csv", index=False)
out.drop(columns=["clean_text"]).to_excel("task2_enriched_dataset.xlsx", index=False)
cluster_summary.to_csv("task2_cluster_summary.csv", index=False)
print("saved")